# Lab 1: Data Exploration and Visualization
**Introduction to Data Science — Florida International University**

**Dataset:** Biscayne Bay Water Quality Sensor Data (`biscayne_bay_water_quality.csv`)

In this lab I explore a real-world marine sensor dataset from Biscayne Bay. I use **pandas** for data manipulation, **Plotly Express** for visualization, and (in a separate `app.py` file) **Streamlit** for an interactive dashboard. The analysis covers descriptive statistics, covariance, correlation, outlier detection via IQR, and two exploratory plots.


## Setup — Import Libraries

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

# Display options so wide DataFrames print cleanly
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)
pd.set_option('display.float_format', lambda x: f'{x:.3f}')

## Step 1 — Load and Explore the Dataset

Load the Biscayne Bay CSV and inspect its structure: shape, column names, data types, a preview of the first rows, and a count of missing values per column.

In [2]:
# Load the dataset
df = pd.read_csv('biscayne_bay_water_quality.csv')

# Shape of the dataset
print(f"Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns\n")

# Column names — important to check before referencing any column,
# because several names contain spaces and unit suffixes in parentheses
print("Columns:")
for c in df.columns:
    print(f"  - {c!r}")

Dataset shape: 942 rows x 10 columns

Columns:
  - 'Latitude'
  - 'Longitude'
  - 'Time'
  - 'Date'
  - 'Total Water Column (m)'
  - 'Vehicle Speed (kn)'
  - 'Salinity (ppt)'
  - 'Temperature (c)'
  - 'pH'
  - 'ODO mg/L'


In [3]:
# Preview the first 5 rows
df.head()

,Latitude,Longitude,Time,Date,Total Water Column (m),Vehicle Speed (kn),Salinity (ppt),Temperature (c),pH,ODO mg/L
0,25.913,-80.138,43:39.1,2/4/22,2.570,0.000,33.650,26.600,7.750,6.480
1,25.913,-80.138,43:40.2,2/4/22,2.580,0.000,35.080,26.500,7.750,6.390
2,25.913,-80.138,43:41.2,2/4/22,2.600,0.040,33.470,26.500,7.750,6.450
3,25.913,-80.138,43:42.2,2/4/22,2.610,0.070,34.100,26.500,7.750,6.400
4,25.913,-80.138,43:43.2,2/4/22,2.600,0.180,35.630,26.500,7.750,6.320


In [4]:
# Data types and non-null counts
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 942 entries, 0 to 941
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Latitude                942 non-null    float64
 1   Longitude               942 non-null    float64
 2   Time                    942 non-null    str    
 3   Date                    942 non-null    str    
 4   Total Water Column (m)  942 non-null    float64
 5   Vehicle Speed (kn)      942 non-null    float64
 6   Salinity (ppt)          942 non-null    float64
 7   Temperature (c)         942 non-null    float64
 8   pH                      942 non-null    float64
 9   ODO mg/L                942 non-null    float64
dtypes: float64(8), str(2)
memory usage: 85.7 KB


In [5]:
# Missing values per column
df.isnull().sum()

Latitude                  0
Longitude                 0
Time                      0
Date                      0
Total Water Column (m)    0
Vehicle Speed (kn)        0
Salinity (ppt)            0
Temperature (c)           0
pH                        0
ODO mg/L                  0
dtype: int64

### Step 1 Observations

The dataset contains **942 observations across 10 columns**. Eight columns are numeric (`float64`): `Latitude`, `Longitude`, `Total Water Column (m)`, `Vehicle Speed (kn)`, `Salinity (ppt)`, `Temperature (c)`, `pH`, and `ODO mg/L` (dissolved oxygen in mg/L). Two columns are stored as objects/strings: `Time` and `Date`. The numeric columns cover the core water-quality variables (Salinity, Temperature, pH, ODO), the sensor's spatial/operational context (Latitude, Longitude, Vehicle Speed, Total Water Column), and time stamps.

**The column names contain spaces and unit suffixes in parentheses** (e.g., `'Salinity (ppt)'`, `'Temperature (c)'`), so I must quote them exactly when referencing them — otherwise pandas will raise a `KeyError`. This is the most likely place for typos to break the rest of the notebook.

**There are zero missing values in any column**, which is unusual for raw sensor data and means no imputation or dropping of rows is required. One thing worth flagging: `Latitude` ranges only from 25.9115 to 25.9128 and `Longitude` from −80.1379 to −80.1369 — both have a standard deviation near 0.0003, meaning the sensor stayed at essentially a single geographic location for the entire dataset. I will exclude Latitude and Longitude from the correlation analysis in Step 3 because their near-zero variance makes any correlation with them statistically meaningless.


## Step 2 — Descriptive Statistics and EDA

Use `df.describe()` to get count, mean, std, min, quartiles, and max for every numeric column. Then look for skew, unusual ranges, and any hint of sensor behavior.

In [6]:
df.describe()

,Latitude,Longitude,Total Water Column (m),Vehicle Speed (kn),Salinity (ppt),Temperature (c),pH,ODO mg/L
count,942.000,942.000,942.000,942.000,942.000,942.000,942.000,942.000
mean,25.912,-80.137,3.096,0.958,37.176,24.172,7.746,5.375
std,0.000,0.000,0.730,0.252,0.971,0.544,0.007,0.232
min,25.912,-80.138,1.060,0.000,33.020,23.600,7.730,5.030
25%,25.912,-80.138,2.790,0.970,36.520,23.800,7.740,5.210
50%,25.912,-80.137,3.260,1.030,37.170,24.000,7.740,5.300
75%,25.913,-80.137,3.598,1.070,37.818,24.400,7.750,5.510
max,25.913,-80.137,4.190,1.380,39.350,26.600,7.770,6.480


In [7]:
# Quick skew check: compare mean vs median for each numeric column
skew_check = pd.DataFrame({
    'mean': df.select_dtypes(include=np.number).mean(),
    'median': df.select_dtypes(include=np.number).median(),
    'std': df.select_dtypes(include=np.number).std()
})
skew_check['mean_minus_median'] = skew_check['mean'] - skew_check['median']
skew_check.round(4)

,mean,median,std,mean_minus_median
Latitude,25.912,25.912,0.000,-0.000
Longitude,-80.137,-80.137,0.000,-0.000
Total Water Column (m),3.096,3.260,0.730,-0.164
Vehicle Speed (kn),0.958,1.030,0.252,-0.072
Salinity (ppt),37.176,37.170,0.972,0.006
Temperature (c),24.172,24.000,0.544,0.172
pH,7.746,7.740,0.007,0.006
ODO mg/L,5.375,5.300,0.232,0.075


### Step 2 Observations

**Observation 1 — pH is extraordinarily tightly clustered, almost a constant.** pH ranges only from **7.73 to 7.77** — a total spread of just 0.04 units — with a standard deviation of **0.0074**. For context, healthy coastal seawater normally varies between ~7.8 and 8.3, and the sensor here appears to record to only two decimal places, so almost every reading falls on one of five discrete values (7.73, 7.74, 7.75, 7.76, 7.77). This narrow range has two consequences: first, the pH column behaves almost like a categorical variable with 5 levels; second, any correlation computed against pH will be noisy because the variable carries very little information. This is unusual enough that it's worth flagging — it could indicate sensor resolution limits rather than true pH stability.

**Observation 2 — Total Water Column and Vehicle Speed are left-skewed, suggesting operational stops.** For `Total Water Column (m)`, the mean (3.10 m) is noticeably below the median (3.26 m), indicating a left tail of shallower readings. For `Vehicle Speed (kn)`, the IQR sits in a narrow band (0.97–1.07 kn) but the minimum is **0.0 kn** — i.e., the sampling vehicle frequently stopped. Combined with the fact that **Temperature only ranges 23.6–26.6 °C (a narrow 3°C band)** and `Date` values in the raw preview are clustered (2/4/22), this looks like a short-duration survey at a fixed location rather than a wide-area or long-term dataset. That context matters for Step 3: correlations here reflect short-window dynamics (e.g., diurnal cycles, tides), not seasonal variation.


## Step 3 — Covariance and Correlation

Covariance tells us the *direction* of a linear relationship but its magnitude depends on the units of the variables. Correlation normalizes it to **[-1, +1]**, so it's what I'll rely on for interpretation. I exclude `Latitude` and `Longitude` because they are effectively constants (std ≈ 0.0003) — any correlation with them would be numerically unreliable.

In [8]:
# Columns for correlation/covariance analysis (drop the near-constant Lat/Long)
analysis_cols = ['Total Water Column (m)', 'Vehicle Speed (kn)',
                 'Salinity (ppt)', 'Temperature (c)', 'pH', 'ODO mg/L']

# Covariance matrix
cov_matrix = df[analysis_cols].cov()
cov_matrix.round(4)

,Total Water Column (m),Vehicle Speed (kn),Salinity (ppt),Temperature (c),pH,ODO mg/L
Total Water Column (m),0.532,0.120,0.199,0.048,0.001,0.035
Vehicle Speed (kn),0.120,0.064,0.118,0.034,0.000,0.004
Salinity (ppt),0.199,0.118,0.944,0.189,0.003,-0.013
Temperature (c),0.048,0.034,0.189,0.296,0.002,0.049
pH,0.001,0.000,0.003,0.002,0.000,-0.000
ODO mg/L,0.035,0.004,-0.013,0.049,-0.000,0.054


In [9]:
# Correlation matrix (Pearson by default)
corr_matrix = df[analysis_cols].corr()
corr_matrix.round(3)

,Total Water Column (m),Vehicle Speed (kn),Salinity (ppt),Temperature (c),pH,ODO mg/L
Total Water Column (m),1.000,0.652,0.281,0.120,0.110,0.209
Vehicle Speed (kn),0.652,1.000,0.480,0.246,0.233,0.066
Salinity (ppt),0.281,0.480,1.000,0.357,0.409,-0.056
Temperature (c),0.120,0.246,0.357,1.000,0.604,0.390
pH,0.110,0.233,0.409,0.604,1.000,-0.002
ODO mg/L,0.209,0.066,-0.056,0.390,-0.002,1.000


In [10]:
# Programmatically find the strongest positive and negative correlations,
# excluding the diagonal of 1.0 and duplicate (mirrored) pairs
corr_pairs = corr_matrix.where(
    ~np.eye(corr_matrix.shape[0], dtype=bool)
).stack()
corr_pairs = corr_pairs[corr_pairs.index.map(lambda x: x[0] < x[1])]

print("Strongest POSITIVE correlations:")
print(corr_pairs.sort_values(ascending=False).head(3).round(3))
print("\nStrongest NEGATIVE correlations:")
print(corr_pairs.sort_values(ascending=True).head(3).round(3))

Strongest POSITIVE correlations:
Total Water Column (m)  Vehicle Speed (kn)   0.652
Temperature (c)         pH                   0.604
Salinity (ppt)          Vehicle Speed (kn)   0.480
dtype: float64

Strongest NEGATIVE correlations:
ODO mg/L  Salinity (ppt)       -0.056
          pH                   -0.002
          Vehicle Speed (kn)    0.066
dtype: float64


In [11]:
# Visualize the correlation matrix as a heatmap
fig_corr = px.imshow(
    corr_matrix.round(2),
    text_auto=True,
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title='Correlation Matrix — Biscayne Bay Water Quality Variables',
    labels=dict(color='Correlation')
)
fig_corr.update_layout(width=750, height=600)
fig_corr.show()

### Step 3 Observations

**Strongest positive correlation: Total Water Column ↔ Vehicle Speed at r ≈ +0.65.** The sampling vehicle moves faster in deeper water and slows in shallower water. This is an **operational** relationship (survey methodology) rather than an environmental one — shallow areas require more careful navigation — so it shouldn't be interpreted as a finding about water chemistry.

**Second-strongest positive, and the most environmentally interesting: Temperature ↔ pH at r ≈ +0.60.** Warmer water in this dataset is associated with higher pH. This runs opposite to the pure-chemistry expectation (CO₂ is more soluble in cold water, which should *lower* pH when cold), but it fits a shallow-bay **biological** explanation: during warmer daytime hours, phytoplankton photosynthesis accelerates, which consumes CO₂ and *raises* pH. Given that the dataset appears to be a short-duration, single-location survey (Step 2), this correlation is most likely picking up the diurnal photosynthesis cycle rather than any seasonal pattern.

**Strongest negative correlation: Salinity ↔ ODO mg/L at r ≈ −0.056.** This is essentially zero — there are **no meaningfully negative correlations** in this dataset. The next closest is pH ↔ ODO at r ≈ 0.00. The absence of negative relationships is itself the finding: in this narrow temperature and pH window, none of the variables move in opposite directions strongly enough to matter.

One more result worth calling out: **Temperature ↔ ODO mg/L is positive (r ≈ +0.39)**, not negative. Textbook aquatic chemistry says warmer water holds *less* dissolved oxygen, so we'd expect a negative correlation. The likely reason we see the opposite here is the same photosynthesis mechanism above: during the warmer daytime hours when temperature rises, phytoplankton also produce oxygen, and that biological oxygen production outweighs the small thermodynamic drop in solubility across this narrow 3°C range.


## Step 4 — Outlier Detection and Treatment (IQR Method)

The IQR method flags values outside **[Q1 − 1.5·IQR, Q3 + 1.5·IQR]**. I'll apply it to all analysis columns, then focus on the columns with the most outliers.

In [12]:
def iqr_outliers(series):
    """Return (Q1, Q3, IQR, lower_bound, upper_bound, outlier_mask)."""
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    mask = (series < lower) | (series > upper)
    return Q1, Q3, IQR, lower, upper, mask

# Summary across all analysis columns
summary_rows = []
for col in analysis_cols:
    Q1, Q3, IQR, lower, upper, mask = iqr_outliers(df[col])
    low_count = (df[col] < lower).sum()
    high_count = (df[col] > upper).sum()
    summary_rows.append({
        'Column': col,
        'Q1': round(Q1, 3),
        'Q3': round(Q3, 3),
        'IQR': round(IQR, 3),
        'Lower Bound': round(lower, 3),
        'Upper Bound': round(upper, 3),
        'Low Outliers': int(low_count),
        'High Outliers': int(high_count),
        'Total Outliers': int(mask.sum())
    })

outlier_summary = pd.DataFrame(summary_rows)
outlier_summary

,Column,Q1,Q3,IQR,Lower Bound,Upper Bound,Low Outliers,High Outliers,Total Outliers
0,Total Water Column (m),2.790,3.598,0.808,1.579,4.809,65,0,65
1,Vehicle Speed (kn),0.970,1.070,0.100,0.820,1.220,89,16,105
2,Salinity (ppt),36.520,37.818,1.297,34.574,39.764,5,0,5
3,Temperature (c),23.800,24.400,0.600,22.900,25.300,0,37,37
4,pH,7.740,7.750,0.010,7.725,7.765,0,9,9
5,ODO mg/L,5.210,5.510,0.300,4.760,5.960,0,22,22


In [13]:
# Deep-dive on Vehicle Speed — the column with the most outliers
col = 'Vehicle Speed (kn)'
Q1, Q3, IQR, lower, upper, mask = iqr_outliers(df[col])

print(f"--- {col} ---")
print(f"Q1 = {Q1:.3f}")
print(f"Q3 = {Q3:.3f}")
print(f"IQR = {IQR:.3f}")
print(f"Lower bound = {lower:.3f}")
print(f"Upper bound = {upper:.3f}")
print(f"Outliers: {mask.sum()} out of {len(df)} ({100*mask.sum()/len(df):.1f}%)")
print(f"  ({(df[col] < lower).sum()} below lower bound, "
      f"{(df[col] > upper).sum()} above upper bound)")
print(f"\nHow many readings are exactly zero? {(df[col] == 0).sum()}")

--- Vehicle Speed (kn) ---
Q1 = 0.970
Q3 = 1.070
IQR = 0.100
Lower bound = 0.820
Upper bound = 1.220
Outliers: 105 out of 942 (11.1%)
  (89 below lower bound, 16 above upper bound)

How many readings are exactly zero? 2


In [14]:
# Box plots side by side so we can compare the shape of each column's outliers
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig_box = make_subplots(
    rows=2, cols=3,
    subplot_titles=analysis_cols
)
for i, col in enumerate(analysis_cols):
    r, c = i // 3 + 1, i % 3 + 1
    fig_box.add_trace(go.Box(y=df[col], name=col, boxpoints='outliers', showlegend=False),
                      row=r, col=c)
fig_box.update_layout(title='Box Plots — Outliers Across All Analysis Columns',
                      height=700, width=950)
fig_box.show()

### Step 4 Observations and Treatment Decision

The outlier counts per column are very uneven:

| Column | Outliers | Notes |
|---|---|---|
| Vehicle Speed (kn) | 105 | 89 below lower bound — mostly stops (speed = 0) |
| Total Water Column (m) | 65 | All low-end — shallower readings |
| Temperature (c) | 37 | All high-end |
| ODO mg/L | 22 | All high-end |
| pH | 9 | All high-end, but the IQR is only 0.01 so this is a sensor-resolution artifact |
| Salinity (ppt) | 5 | Low-end only |

**Decision: retain the environmental outliers; do not remove them. Exclude or deprioritize `Vehicle Speed (kn)` outliers, because they are operational, not environmental.** My reasoning has two parts.

*For Vehicle Speed:* the 105 "outliers" are overwhelmingly readings where the sampling vehicle was stopped (speed near 0). These are not sensor errors and not water-quality events — they're just periods when the survey platform was stationary. They shouldn't be "treated" at all because they don't describe water. If the analysis were about water chemistry only, I would simply drop `Vehicle Speed (kn)` from the correlation and outlier workflow rather than tinker with its values.

*For Temperature, ODO, Salinity, and pH:* these outliers almost certainly reflect real environmental variation within this short survey window. Temperature high-end outliers (above 25.3°C) line up with the warmer-afternoon portion of a diurnal cycle, and ODO high-end outliers line up with the same window (matching the positive Temperature–ODO correlation from Step 3). The pH "outliers" are a statistical artifact: because the IQR is only 0.01 (the sensor's rounding floor), any reading above 7.765 trips the bound, so flagging them as anomalies is a weakness of the IQR method when applied to a near-constant column — not a real signal. I would keep all of these rows intact. For a modeling downstream, a log transform isn't needed (no strong right skew), but I would consider using **robust statistics** (median, MAD) rather than mean/std for pH because of how little it varies.


## Step 5 — Data Visualization using Plotly Express

Two plots required by the lab: a **scatter plot** of Salinity vs. Temperature, and a **histogram** of pH. Both use the exact column names from the file (with units in parentheses), and both include title and axis labels for full credit.

In [15]:
# Scatter plot: Salinity vs Temperature, colored by ODO (dissolved oxygen)
# Coloring by ODO lets us see the Temperature-ODO relationship from Step 3 visually
fig_scatter = px.scatter(
    df,
    x='Salinity (ppt)',
    y='Temperature (c)',
    color='ODO mg/L',
    color_continuous_scale='Viridis',
    title='Salinity vs. Temperature in Biscayne Bay (colored by Dissolved Oxygen)',
    labels={
        'Salinity (ppt)': 'Salinity (ppt)',
        'Temperature (c)': 'Temperature (°C)',
        'ODO mg/L': 'Dissolved Oxygen (mg/L)'
    },
    opacity=0.65
)
fig_scatter.update_layout(width=850, height=550)
fig_scatter.show()

In [16]:
# Histogram of pH — use fewer bins because pH only takes ~5 discrete values
fig_hist = px.histogram(
    df,
    x='pH',
    nbins=20,
    title='Distribution of pH Readings in Biscayne Bay',
    labels={'pH': 'pH', 'count': 'Frequency'},
    color_discrete_sequence=['#2E86AB']
)
fig_hist.add_vline(
    x=df['pH'].mean(),
    line_dash='dash',
    line_color='red',
    annotation_text=f"Mean = {df['pH'].mean():.3f}",
    annotation_position='top'
)
fig_hist.update_layout(width=850, height=500, bargap=0.15)
fig_hist.show()

### Step 5 Observations

**Scatter plot (Salinity vs. Temperature, colored by ODO):** The points form a loose upward-sloping cloud, consistent with the moderate positive correlation from Step 3 (r ≈ +0.36) — as salinity rises from ~33 ppt toward ~39 ppt, temperature tends to rise from ~23.6°C toward ~26.6°C. The relationship is noisy, not a tight line, so salinity alone is not a good predictor of temperature. The more striking pattern is the **color gradient**: the yellow (high ODO) points cluster at the upper-right of the plot (warm + salty), while the dark purple (low ODO) points cluster at the lower-left (cool + fresher). This visual tri-variate pattern confirms the Temperature–ODO positive correlation from Step 3 and suggests that the three variables (Salinity, Temperature, ODO) move together, likely all driven by a shared time-of-day / tidal cycle across the short survey window.

**Histogram (pH):** The distribution is **extremely narrow and discrete** — the x-axis spans only 7.73 to 7.77, a total of 0.04 pH units, and the readings pile up on just five values (7.73, 7.74, 7.75, 7.76, 7.77). The mode is **7.74**, with the mean (red dashed line, 7.746) sitting just slightly above it. This is not a smooth bell curve; it's a bar chart of sensor-resolution-limited values. Two takeaways: (1) this dataset's pH sensor is clearly reporting only to two decimal places, so the "distribution" is a reflection of sensor precision as much as of chemistry, and (2) the bay's pH was genuinely near-constant during the survey window, consistent with a short, single-location dataset rather than one capturing big environmental swings.


## Step 6 — Streamlit App

The interactive app is in a separate file (**`app.py`**) because Streamlit apps launch via a CLI command, not from inside a notebook cell.

**To run it:**
```bash
streamlit run app.py
```

### Features implemented (per rubric)
1. **Dataset preview toggle** — `st.checkbox()` to show/hide the raw data
2. **Descriptive statistics** — `df.describe()` displayed
3. **Correlation matrix** — rendered as a Plotly heatmap (with Lat/Long excluded for meaningful interpretation)
4. **Interactive charts** — dropdowns let the user pick X and Y variables for a scatter plot, plus a slider to adjust histogram bin count

See `app.py` for the source code.

---

## Conclusion

This lab walked through the full data-exploration pipeline on the Biscayne Bay water-quality dataset (942 observations, 10 columns): loading and inspecting structure, computing descriptive statistics, analyzing covariance and correlation, detecting outliers via the IQR method, visualizing patterns with Plotly Express, and building a Streamlit app for interactive exploration.

The most environmentally interesting finding is the **strong positive correlation between Temperature and pH (r ≈ +0.60)**, together with a **positive correlation between Temperature and ODO (r ≈ +0.39)**. Both run counter to the simple thermodynamic expectation for water chemistry and together point to the **photosynthesis / diurnal cycle** as the dominant signal in this short, single-location survey — not seasonal temperature effects or depth-driven stratification. The strongest correlation overall (Total Water Column ↔ Vehicle Speed, r ≈ +0.65) is an operational artifact of how the survey was conducted and should not be treated as an environmental finding.

The dataset is clean (no missing values), spatially constant (Lat/Long essentially fixed), and pH is near-constant at sensor-resolution precision, which limits how much pH can contribute to downstream modeling. For any future analysis, I would treat this as **time-series data from a single station** rather than a survey, and use temporal features (from the `Time` column) as the main explanatory axis.
